# Numbers & Tables -- "Heterogeneity Gap and Failure-Mode Analysis"

Generates every number and table for the first results section. Loads on-disk
seed-evaluation artifacts and reduces them with the scripts in
`src/analysis/utils` (`compute_normalized`, `compute_k`, `paired_difference`,
`holm_bonferroni`).

**Workflow:** run top-to-bottom on data that already exists. The **lead
reporting metric is the optimality gap to the BS-Optimized ceiling** (`gap%`,
from `compute_gap`); the pre-registered **normalized score is retained** in the
same tables and is the unit the paired tests run in. The **noPS axis is cut**, so
the localization table is the centralized-critic contrast (MAPPO-PS vs IPPO-PS)
plus the gap to the CPPO and BS-Optimized ceilings.

Outputs (CSV + LaTeX) land in `experiment_outputs/analysis_outputs/`.

In [ ]:
# === SETUP: paths + the one thing you paste ==================================
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")

# Notebook lives in src/<analysis|visualization>/, so repo root is two parents up.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Outputs are written here, never into thesis/ (copy them over yourself).
OUT_DIR = REPO_ROOT / "experiment_outputs" / "analysis_outputs"
FIG_DIR = REPO_ROOT / "experiment_outputs" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- Environments -----------------------------------------------------------
ENVS   = ["symmetric", "sku_hetero", "demand_hetero", "cost_hetero", "combined_hetero"]
HETERO = ["sku_hetero", "demand_hetero", "cost_hetero", "combined_hetero"]
ENV_TOK   = {"symmetric": "Symmetric", "sku_hetero": "SKUHetero",
             "demand_hetero": "DemandHetero", "cost_hetero": "CostHetero",
             "combined_hetero": "CombinedHetero"}
ENV_LABEL = {"symmetric": "Symmetric", "sku_hetero": "SKU", "demand_hetero": "Demand",
             "cost_hetero": "Cost", "combined_hetero": "Combined"}

# ---- Algorithms in the story (noPS axis cut) --------------------------------
# CPPO is the centralized RL ceiling. Re-add "MAPPO-noPS"/"IPPO-noPS" here only
# if you later run them on the hetero envs.
ALGOS = ["MAPPO-PS", "IPPO-PS", "CPPO"]
BASELINE_KINDS = ["Random", "ConstantOrder", "BSNewsvendor", "BSAdaptive",
                  "BSOptimized", "BSIndependent"]

# ====== PASTE HERE ===========================================================
# Re-tuned MAPPO-PS rollouts produced with run_evaluation.sh --visualize.
# Each value is a directory that DIRECTLY contains episodes_data.npz
# (E1.5 locked MAPPO-PS per env; symmetric is the E0.3 reference column).
# Leave None until ready -> dependent cells skip cleanly.
ROLLOUTS = {
    "symmetric":       None,   # E0.3 locked MAPPO-PS  (reference column)
    "sku_hetero":      None,   # E1.5 SKUHetero        (MAIN: per-SKU collapse)
    "demand_hetero":   None,   # E1.5 DemandHetero     (MAIN: per-warehouse scale)
    "cost_hetero":     None,   # E1.5 CostHetero       (appendix)
    "combined_hetero": None,   # E1.5 CombinedHetero   (appendix)
}

# ---- Directory registry (pre-filled to the on-disk E0/E1 layout) ------------
TUNING = REPO_ROOT / "experiment_outputs" / "Tuning"

def _locked(seed_eval_dir):
    """load_run config dir: locked '01_*' trial (tune mode) else the dir itself
    (single-config / transfer mode)."""
    d = Path(seed_eval_dir)
    if not d.is_dir():
        return None
    ranked = sorted(d.glob("01_*"))
    return ranked[0] if ranked else d

_E0 = TUNING / "E0"
SYMM_RL = {
    "MAPPO-PS":   _E0 / "MAPPO_PS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"   / "seed_evaluation",
    "IPPO-PS":    _E0 / "IPPO_PS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"    / "seed_evaluation",
    "CPPO":       _E0 / "CPPO_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"       / "seed_evaluation",
    "MAPPO-noPS": _E0 / "MAPPO_NoPS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0" / "seed_evaluation",
    "IPPO-noPS":  _E0 / "IPPO_NoPS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"  / "seed_evaluation",
}
SYMM_BASE = _E0 / "BASELINES_Tune_3WH_5SKUS_Symmetric_E0" / "seed_evaluation"

_E11 = TUNING / "E1" / "E1.1_Hetero_Transfer"
_TG = {"MAPPO-PS": "MAPPO_PS", "IPPO-PS": "IPPO_PS", "CPPO": "CPPO"}
def transfer_dir(algo, env):
    a = _TG.get(algo)
    if a is None:
        return None
    return (_E11 / f"{a}_HeteroTransfer_E1.1"
                 / f"{a}_SeedEval_3WH_5SKUS_{ENV_TOK[env]}_E1.1" / "seed_evaluation")

def retune_dir(algo, env):
    tok = ENV_TOK[env]
    if algo == "CPPO":
        return TUNING/"E1"/"E1.3_Hetero_CPPO_Retune"/f"CPPO_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.3"/"seed_evaluation"
    if algo == "IPPO-PS":
        return TUNING/"E1"/"E1.4_Hetero_IPPO_Retune"/f"IPPO_PS_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.4"/"seed_evaluation"
    if algo == "MAPPO-PS":
        return TUNING/"E1"/"E1.5_Hetero_MAPPO_Retune"/f"MAPPO_PS_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.5"/"seed_evaluation"
    return None

_E12 = TUNING/"E1"/"E1.2_Hetero_Baselines"/"BASELINES_3WH5SKUS_Hetero_E1.2"
def baseline_dir(env, kind):
    if env == "symmetric":
        return SYMM_BASE / kind
    return _E12 / f"BASELINES_3WH5SKUS_{ENV_TOK[env]}" / "seed_evaluation" / kind

print("REPO_ROOT :", REPO_ROOT)
print("tables ->", OUT_DIR)
print("figures ->", FIG_DIR)


In [ ]:
# === LOAD: read every run once (graceful about missing dirs) =================
from src.visualization.utils.plot_data_loader import load_run, load_episodes

def _resolve(p):
    p = Path(p)
    return _locked(p) if p.name == "seed_evaluation" else p

def _try_load(p, what):
    if p is None:
        return None
    rp = _resolve(p)
    if rp is None or not rp.is_dir():
        return None
    try:
        run = load_run(rp)
    except Exception as e:
        print(f"  ! load failed [{what}]: {e}")
        return None
    return run if run["seed_data"] else None

RL_SYMM     = {a: _try_load(SYMM_RL.get(a), f"symm/{a}") for a in ALGOS}
RL_TRANSFER = {e: {a: _try_load(transfer_dir(a, e), f"transfer/{a}/{e}") for a in ALGOS} for e in HETERO}
RL_RETUNE   = {e: {a: _try_load(retune_dir(a, e),   f"retune/{a}/{e}")   for a in ALGOS} for e in HETERO}
BASE        = {e: {k: _try_load(baseline_dir(e, k), f"base/{k}/{e}")     for k in BASELINE_KINDS} for e in ENVS}

def _n(run):
    return len(run["seed_data"]) if run else 0

print("== coverage (n seeds; '-' = missing) ==")
print("symmetric RL:", {a: _n(RL_SYMM[a]) or "-" for a in ALGOS})
for e in HETERO:
    print(f"{ENV_LABEL[e]:9s} transfer:", {a: _n(RL_TRANSFER[e][a]) or "-" for a in ALGOS},
          " retune:", {a: _n(RL_RETUNE[e][a]) or "-" for a in ALGOS})
for e in ENVS:
    print(f"{ENV_LABEL[e]:9s} baselines:", {k: _n(BASE[e][k]) or "-" for k in BASELINE_KINDS})


## 1. Reference anchors and operational coupling κ

In [ ]:
# --- Reference anchors (R_Random, R_BS-Optimized) + operational coupling kappa
import numpy as np
import pandas as pd
from src.analysis.utils.compute_normalized import reference_value, normalize_run
from src.analysis.utils.compute_k import compute_kappa

anchors = {}
for env in ENVS:
    rnd, bso = BASE[env].get("Random"), BASE[env].get("BSOptimized")
    if rnd and bso:
        anchors[env] = (reference_value(rnd), reference_value(bso))

anchors_df = pd.DataFrame(
    [{"env": ENV_LABEL[e], "R_Random": anchors[e][0], "R_BS-Optimized": anchors[e][1]}
     for e in ENVS if e in anchors]
).set_index("env")
print("Reference anchors (across-seed mean return):")
display(anchors_df.round(2))

# kappa: cross-region fulfillment fraction under BS-Optimized, pooled over all
# saved evaluation episodes/seeds. Uses the shipment tensors already in each
# baseline episodes_data.npz (no re-rollout needed). Fills the env-design tables.
kappa_recs = []
for env in ENVS:
    bso = BASE[env].get("BSOptimized")
    if not bso:
        continue
    eps = []
    for sd in bso["seed_data"].values():
        try:
            eps.extend(load_episodes(sd["seed_run_dir"]))
        except FileNotFoundError:
            pass
    if not eps:
        print(f"  ! no BS-Optimized episodes for {env}")
        continue
    k = compute_kappa(eps, by_warehouse=True)
    rec = {"env": ENV_LABEL[env], "kappa": k["kappa_pooled"],
           "kappa_ep_mean": k["kappa_mean"], "kappa_ep_std": k["kappa_std"],
           "n_episodes": k["n_episodes"]}
    kw = k.get("kappa_pooled_by_warehouse")
    if kw is not None:
        for i, v in enumerate(np.asarray(kw, dtype=float)):
            rec[f"kappa_WH{i}"] = float(v)
    kappa_recs.append(rec)

kappa_df = pd.DataFrame(kappa_recs).set_index("env")
print("\nOperational coupling kappa (BS-Optimized):")
display(kappa_df.round(4))

## 2. Symmetric reference (the control is solved)

In [ ]:
# --- Table: symmetric reference (the control is solved) ----------------------
# Lead metric: gap_pct = % above the BS-Optimized cost (0 = ceiling). Normalized
# score and raw return retained alongside.
from src.analysis.utils.compute_gap import gap_run

def raw_final(run):
    vals = [s["final_eval"]["episode_return_mean"] for s in run["seed_data"].values()
            if s["final_eval"] and s["final_eval"].get("episode_return_mean") is not None]
    if not vals:
        return (np.nan, np.nan, 0)
    return (float(np.mean(vals)),
            float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            len(vals))

env = "symmetric"
rows = []
if env in anchors:
    r_rand, r_bs = anchors[env]
    def _row(name, run):
        m, sd, n = raw_final(run)
        ns = normalize_run(run, r_rand, r_bs)
        g = gap_run(run, r_bs)
        return {"condition": name,
                "gap_pct": 100 * g["mean"], "gap_pct_sem": 100 * g["sem"],
                "return_mean": m, "return_std": sd,
                "norm_score": ns["mean"], "norm_sem": ns["sem"], "n": n}
    for a in ALGOS:
        if RL_SYMM.get(a):
            rows.append(_row(a, RL_SYMM[a]))
    for k in BASELINE_KINDS:
        if BASE[env].get(k):
            rows.append(_row(k, BASE[env][k]))

sym_df = pd.DataFrame(rows).set_index("condition")
display(sym_df.round({"gap_pct": 2, "gap_pct_sem": 2, "return_mean": 1,
                      "return_std": 1, "norm_score": 3, "norm_sem": 3}))

## 3. The heterogeneity gap: transferred vs re-tuned

Lead metric is the optimality gap to the BS-Optimized ceiling (`gap% = (R_BS - R)/|R_BS|`); `gap%_retune` is the residual gap. `frac_gap_closed = (gap_transfer - gap_retune) / gap_transfer` is the share of the transferred optimality gap removed by per-env re-tuning (pre-registered expectation < 0.5: the residual is architectural). Normalized score is kept in the last two columns as the registered unit.

In [ ]:
# --- Table: heterogeneity gap (optimality gap to ceiling; NS retained) -------
# gap%_retune IS the residual gap to the BS-Optimized ceiling (0 = matched).
# frac_gap_closed = share of the transferred optimality gap removed by re-tuning.
def gap_mean_sem(run, env):
    if not run or env not in anchors:
        return (np.nan, np.nan)
    g = gap_run(run, anchors[env][1])   # BS-Optimized anchor only
    return (g["mean"], g["sem"])

def ns_mean(run, env):
    if not run or env not in anchors:
        return np.nan
    return normalize_run(run, *anchors[env])["mean"]

gap_rows = []
for env in HETERO:
    for a in ALGOS:
        gt, _     = gap_mean_sem(RL_TRANSFER[env].get(a), env)
        gq, gq_se = gap_mean_sem(RL_RETUNE[env].get(a), env)
        closed = ((gt - gq) / gt) if (np.isfinite(gt) and np.isfinite(gq) and gt != 0) else np.nan
        gap_rows.append({
            "env": ENV_LABEL[env], "algo": a,
            "gap%_transfer": 100 * gt, "gap%_retune": 100 * gq, "gap%_retune_sem": 100 * gq_se,
            "frac_gap_closed": closed,
            "NS_transfer": ns_mean(RL_TRANSFER[env].get(a), env),
            "NS_retune": ns_mean(RL_RETUNE[env].get(a), env),
        })

gap_df = pd.DataFrame(gap_rows).set_index(["env", "algo"])
display(gap_df.round({"gap%_transfer": 2, "gap%_retune": 2, "gap%_retune_sem": 2,
                      "frac_gap_closed": 3, "NS_transfer": 3, "NS_retune": 3}))

## 4. E1 primary tests (re-tuned vs transferred)

Paired differences per algorithm across the four hetero envs, Holm-Bonferroni corrected within each algorithm's family (k = 4).

In [ ]:
# --- Table: E1 primary tests (re-tuned vs transferred), Holm per algo --------
from src.analysis.utils.paired_difference import paired_difference
from src.analysis.utils.holm_bonferroni import holm_bonferroni

primary_rows = []
for a in ALGOS:
    fam, objs = {}, {}
    for env in HETERO:
        t, q = RL_TRANSFER[env].get(a), RL_RETUNE[env].get(a)
        if not (t and q and env in anchors):
            continue
        pa = normalize_run(q, *anchors[env])["per_seed"]   # A = re-tuned
        pb = normalize_run(t, *anchors[env])["per_seed"]   # B = transferred
        res = paired_difference(pa, pb, label=f"{a}/{ENV_LABEL[env]}")
        fam[env], objs[env] = res.p_value, res
    holm = holm_bonferroni(fam) if fam else {}
    for env, res in objs.items():
        h = holm[env]
        primary_rows.append({
            "algo": a, "env": ENV_LABEL[env], "delta": res.delta,
            "ci_low": res.ci_low, "ci_high": res.ci_high, "d_z": res.cohens_dz,
            "p": res.p_value, "p_holm": h.p_adjusted, "reject_holm": h.reject, "n": res.n,
        })

primary_df = (pd.DataFrame(primary_rows).set_index(["algo", "env"])
              if primary_rows else pd.DataFrame())
display(primary_df.round(4) if not primary_df.empty else "no paired data yet")

## 5. Localizing the gap (noPS cut)

Centralized-critic effect and decentralization cost on the re-tuned policies.

In [ ]:
# --- Table: localization without noPS ----------------------------------------
# Centralized-critic effect (MAPPO-PS vs IPPO-PS) and cost of decentralization
# (MAPPO-PS vs CPPO ceiling), both on the re-tuned policies, paired by seed.
critic_rows = []
for env in HETERO:
    if env not in anchors:
        continue
    m = RL_RETUNE[env].get("MAPPO-PS")
    i = RL_RETUNE[env].get("IPPO-PS")
    c = RL_RETUNE[env].get("CPPO")
    pm = normalize_run(m, *anchors[env])["per_seed"] if m else None
    for other, name in [(i, "MAPPO-PS - IPPO-PS"), (c, "MAPPO-PS - CPPO")]:
        if pm is None or other is None:
            continue
        po = normalize_run(other, *anchors[env])["per_seed"]
        r = paired_difference(pm, po, label=f"{name} / {ENV_LABEL[env]}")
        critic_rows.append({"contrast": name, "env": ENV_LABEL[env], "delta": r.delta,
                            "ci_low": r.ci_low, "ci_high": r.ci_high,
                            "d_z": r.cohens_dz, "p": r.p_value, "n": r.n})

critic_df = (pd.DataFrame(critic_rows).set_index(["contrast", "env"])
             if critic_rows else pd.DataFrame())
display(critic_df.round(4) if not critic_df.empty else "no re-tuned pairs yet")

## 6. Export

In [ ]:
# --- Export every table to CSV + LaTeX (booktabs) ----------------------------
def save_table(df, name):
    if df is None or (hasattr(df, "empty") and df.empty):
        print(f"  (skip {name}: empty)")
        return
    df.to_csv(OUT_DIR / f"{name}.csv")
    try:
        with open(OUT_DIR / f"{name}.tex", "w", encoding="utf-8") as f:
            f.write(df.to_latex(float_format=lambda x: f"{x:.3f}", escape=True))
    except Exception as e:
        print(f"  (latex skip {name}: {e})")
    print(f"  saved {name}")

for name, df in [
    ("anchors", anchors_df), ("kappa", kappa_df), ("symmetric_reference", sym_df),
    ("heterogeneity_gap", gap_df), ("e1_primary_tests", primary_df),
    ("critic_contrast", critic_df),
]:
    try:
        save_table(df, name)
    except NameError:
        print(f"  (skip {name}: not computed)")
print("done ->", OUT_DIR)